## What this notebook is

This validates one claim from the analysis in this repository.

The party-switch check compares, for each state where the governor's party changed, average birth weight in the last year under the old party against the first year under the new one. It is built by `models/marts/fct_party_switch_effect.sql`. The argument originally made from it was that birth weight declined regardless of which direction the switch went, which is indirect evidence against a real party effect. Reasonable, but it infers the absence of an effect from a pattern rather than measuring the effect and finding it small.

This notebook measures it instead, with **difference-in-differences**. Every switching state's before-and-after change is compared against the same-period change in states that did not switch, so the national decline that was happening anyway, 3,267g to 3,242g across the study period, is netted out rather than charged to the governor.

**Data access:** the tables queried below are the ones `dbt run` builds from the models in this repository, read from BigQuery through the `bq` command-line tool. Rebuild them in your own project and the queries work unchanged apart from the project name.

### Q1 — What did the original analysis actually compute?
Keeping this visible, not silently replacing it — the fix should be legible against what it's fixing.

In [1]:
import subprocess
import io
import numpy as np
import pandas as pd
from scipy import stats

def run_bq(sql):
    result = subprocess.run(
        ['bq', 'query', '--use_legacy_sql=false', '--format=csv', '--max_rows=200000', sql],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"BigQuery error:\n{result.stderr}")
    return pd.read_csv(io.StringIO(result.stdout))

naive = run_bq('''
    SELECT state, before_year, after_year, party_before, party_after,
           switch_direction, birth_weight_before, birth_weight_after, birth_weight_change
    FROM `dk-data-analytics.birth_outcomes.fct_party_switch_effect`
    ORDER BY state, before_year
''')
print(f"{len(naive)} switch events (21 states, Nevada switched twice)")
naive


22 switch events (21 states, Nevada switched twice)


,state,before_year,after_year,party_before,party_after,switch_direction,birth_weight_before,birth_weight_after,birth_weight_change
0,Alaska,2018,2019,Independent,Republican,Independent → Republican,3409.500429,3394.467871,-15.032558
1,Arizona,2022,2023,Republican,Democratic,Republican → Democratic,3260.595938,3259.527549,-1.068389
2,Illinois,2018,2019,Republican,Democratic,Republican → Democratic,3262.023473,3257.309624,-4.713849
3,Kansas,2018,2019,Republican,Democratic,Republican → Democratic,3301.285048,3291.297246,-9.987802
4,Kentucky,2019,2020,Republican,Democratic,Republican → Democratic,3254.519962,3261.701324,7.181362
5,Louisiana,2023,2024,Democratic,Republican,Democratic → Republican,3146.306180,3140.392366,-5.913813
6,Maine,2018,2019,Republican,Democratic,Republican → Democratic,3344.759947,3341.197202,-3.562744
7,Maryland,2022,2023,Republican,Democratic,Republican → Democratic,3254.330285,3245.884014,-8.446271
8,Massachusetts,2022,2023,Republican,Democratic,Republican → Democratic,3280.854718,3279.638183,-1.216536
9,Michigan,2018,2019,Republican,Democratic,Republican → Democratic,3279.287661,3265.181228,-14.106433


**The flaw:** each row above is a single state's before-vs-after comparison — no reference point for what would have happened in that same window *without* a switch. A national decline in birth weight, happening for reasons that have nothing to do with any governor, would show up here as if the party switch caused it.

### Q2 — Rebuild the birth-weighted state-year baseline
Same weighting logic as the rest of this analysis: `SUM(avg_birth_weight_grams * births) / SUM(births)`, not a simple mean — avoids small-count cells distorting a state's yearly average.

In [2]:
state_year = run_bq('''
    SELECT
        state,
        year,
        SUM(avg_birth_weight_grams * births) / SUM(births) AS weighted_avg_birth_weight,
        SUM(births) AS total_births
    FROM `dk-data-analytics.birth_outcomes.fct_birth_outcomes`
    WHERE avg_birth_weight_grams IS NOT NULL
      AND NOT is_suppressed
    GROUP BY state, year
    ORDER BY state, year
''')
print(f"{len(state_year)} state-year rows, {state_year['state'].nunique()} states, "
      f"{state_year['year'].min()}-{state_year['year'].max()}")
state_year.head()


459 state-year rows, 51 states, 2016-2024


,state,year,weighted_avg_birth_weight,total_births
0,Alabama,2016,3192.624736,59151
1,Alabama,2017,3189.104745,58941
2,Alabama,2018,3184.370174,57761
3,Alabama,2019,3181.844656,58615
4,Alabama,2020,3179.070672,57647


In [3]:
political = run_bq('''
    SELECT state_name AS state, year, governor_party
    FROM `dk-data-analytics.birth_outcomes.political_control_by_state_year`
    ORDER BY state, year
''')
print(f"{len(political)} state-year political control rows")
political.head()


459 state-year political control rows


,state,year,governor_party
0,Alabama,2016,Republican
1,Alabama,2017,Republican
2,Alabama,2018,Republican
3,Alabama,2019,Republican
4,Alabama,2020,Republican


### Q3 — For each switch event, who's a valid control?
A control state is one whose governor's party was the *same* at both the before-year and after-year of this specific event's window — meaning no switch of its own happened during that same period. This naturally excludes any other state that happened to switch during the same window without needing a special case for it.

In [4]:
def get_weighted_bw(state, year):
    row = state_year[(state_year['state'] == state) & (state_year['year'] == year)]
    if row.empty:
        return None, None
    return row['weighted_avg_birth_weight'].iloc[0], row['total_births'].iloc[0]

def get_party(state, year):
    row = political[(political['state'] == state) & (political['year'] == year)]
    return row['governor_party'].iloc[0] if not row.empty else None

results = []
for _, ev in naive.iterrows():
    treated_state = ev['state']
    y0, y1 = ev['before_year'], ev['after_year']

    # Valid controls: party unchanged for this state across this exact window, and not the treated state
    all_states = political['state'].unique()
    controls = []
    for s in all_states:
        if s == treated_state:
            continue
        p0, p1 = get_party(s, y0), get_party(s, y1)
        if p0 is not None and p0 == p1:
            controls.append(s)

    # Pooled, birth-weighted control-group change over the same window
    before_vals, after_vals = [], []
    for s in controls:
        bw0, n0 = get_weighted_bw(s, y0)
        bw1, n1 = get_weighted_bw(s, y1)
        if bw0 is not None and bw1 is not None:
            before_vals.append((bw0, n0))
            after_vals.append((bw1, n1))

    pooled_before = sum(bw * n for bw, n in before_vals) / sum(n for _, n in before_vals)
    pooled_after = sum(bw * n for bw, n in after_vals) / sum(n for _, n in after_vals)
    control_change = pooled_after - pooled_before

    treated_change = ev['birth_weight_change']
    did = treated_change - control_change

    results.append({
        'state': treated_state, 'before_year': y0, 'after_year': y1,
        'switch_direction': ev['switch_direction'],
        'n_controls': len(before_vals),
        'treated_change': treated_change,
        'control_change': control_change,
        'diff_in_diff': did,
    })

did_df = pd.DataFrame(results)
print(f"Control state count per event — min: {did_df['n_controls'].min()}  "
      f"max: {did_df['n_controls'].max()}  median: {did_df['n_controls'].median():.0f}")
did_df


Control state count per event — min: 43  max: 50  median: 46


,state,before_year,after_year,switch_direction,n_controls,treated_change,control_change,diff_in_diff
0,Alaska,2018,2019,Independent → Republican,43,-15.032558,-7.094377,-7.938181
1,Arizona,2022,2023,Republican → Democratic,47,-1.068389,-0.407039,-0.661350
2,Illinois,2018,2019,Republican → Democratic,43,-4.713849,-7.094377,2.380528
3,Kansas,2018,2019,Republican → Democratic,43,-9.987802,-7.094377,-2.893425
4,Kentucky,2019,2020,Republican → Democratic,50,7.181362,1.309091,5.872271
5,Louisiana,2023,2024,Democratic → Republican,50,-5.913813,0.860892,-6.774706
6,Maine,2018,2019,Republican → Democratic,43,-3.562744,-7.094377,3.531633
7,Maryland,2022,2023,Republican → Democratic,47,-8.446271,-0.407039,-8.039232
8,Massachusetts,2022,2023,Republican → Democratic,47,-1.216536,-0.407039,-0.809497
9,Michigan,2018,2019,Republican → Democratic,43,-14.106433,-7.094377,-7.012056


### Q4 — Across all 22 events, is the average diff-in-diff distinguishable from zero?

In [5]:
gaps = did_df['diff_in_diff'].values

print(f"Mean diff-in-diff: {gaps.mean():+.3f} g")
print(f"SD: {gaps.std(ddof=1):.3f} g")
print(f"Range: {gaps.min():+.3f} to {gaps.max():+.3f} g")

t_stat, p_value = stats.ttest_1samp(gaps, popmean=0)
print(f"\nPaired t-test vs. zero: t={t_stat:.3f}  p={p_value:.4f}")

w_stat, w_p = stats.wilcoxon(gaps)
print(f"Wilcoxon signed-rank (less sensitive to any single extreme event): W={w_stat:.1f}  p={w_p:.4f}")


Mean diff-in-diff: -2.392 g
SD: 5.339 g
Range: -15.105 to +6.042 g

Paired t-test vs. zero: t=-2.101  p=0.0479
Wilcoxon signed-rank (less sensitive to any single extreme event): W=70.0  p=0.0684


### Q5 — Direct result vs. the original indirect argument
The slide's falsification argument (birth weight declines regardless of switch direction) was indirect evidence against a real party effect. This is the direct measurement.

In [6]:
print("Naive (original) mean before/after change across 22 events:",
      f"{naive['birth_weight_change'].mean():+.3f} g")
print("Diff-in-diff (trend-adjusted) mean change across 22 events:   ",
      f"{gaps.mean():+.3f} g")
print(f"\nHow much of the naive change was actually just the shared trend: "
      f"{did_df['control_change'].mean():+.3f} g average control-state change over the same windows")


Naive (original) mean before/after change across 22 events: -6.633 g
Diff-in-diff (trend-adjusted) mean change across 22 events:    -2.392 g

How much of the naive change was actually just the shared trend: -4.241 g average control-state change over the same windows


### Conclusion
The naive before/after estimate (−6.6g) was almost twice the size of the trend-adjusted one (−2.4g) — most of what looked like a party-switch effect was actually just the shared national decline that non-switching states experienced over the same windows too. That's the core value diff-in-differences adds here: it separates "this state changed" from "this state changed more or less than everywhere else did anyway."

But the residual −2.4g isn't cleanly zero either, and the two significance tests disagree at the conventional 0.05 line: the paired t-test comes in just under it (p=0.048), the Wilcoxon signed-rank test — less sensitive to any single extreme event — comes in just over it (p=0.068). A result that flips depending on which reasonable test is used isn't strong evidence in either direction; it's evidence of a small, genuinely uncertain effect, not a confidently confirmed or confidently ruled-out one.

**Revised takeaway, replacing the original slide's argument:** the direction-invariance argument was directionally right — most of the apparent decline around party switches wasn't really about party. But "no effect" overstated it. The honest finding is: a small, borderline-significant residual decline remains after controlling for the shared trend, on the order of ~2 grams, too small to act on with confidence but too present to declare fully absent. That's a real, defensible, appropriately-hedged finding — a direct measurement replacing an indirect argument, landing close to the same place but with an honest confidence interval around it instead of a flat yes/no.